# Project 27 — Freshness-Aware News RAG

## Prioritize Recent Documents in Retrieval

**Goal:** Blend semantic relevance with document recency so time-sensitive
queries surface the latest information.

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

### What You'll Learn
1. Timestamped document indexing
2. Exponential freshness decay scoring
3. Configurable relevance-vs-freshness weighting
4. Comparison of freshness-blind vs freshness-aware retrieval

### Prerequisites
- **Ollama** with `nomic-embed-text` and `qwen3:8b`

In [ ]:
!pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    print(f"Ollama running — {len(r.json().get('models',[]))} model(s)")
except Exception as e:
    print(f"Cannot reach Ollama: {e}")

## Step 2 — Timestamped News Corpus

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import math

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
emb = OllamaEmbeddings(model="nomic-embed-text")

articles = [
    Document(page_content="AI startup raises $50M Series B to build enterprise RAG solutions. "
        "Plans to hire 100 engineers and expand to Europe.",
        metadata={"days_ago": 1, "source": "techcrunch"}),
    Document(page_content="New open-source embedding model achieves state-of-the-art on MTEB "
        "benchmark. Available on HuggingFace with Apache 2.0 license.",
        metadata={"days_ago": 7, "source": "arxiv"}),
    Document(page_content="AI startup raises $20M Series A for enterprise RAG platform. "
        "Focuses on healthcare and legal verticals with HIPAA compliance.",
        metadata={"days_ago": 90, "source": "techcrunch"}),
    Document(page_content="Ollama releases version 0.5 with multi-model support and improved "
        "performance. Memory usage reduced by 30%.",
        metadata={"days_ago": 3, "source": "github"}),
    Document(page_content="Study shows RAG systems reduce hallucination by 70% compared to "
        "base LLMs in enterprise settings. Based on 500 queries.",
        metadata={"days_ago": 30, "source": "research"}),
    Document(page_content="Major cloud provider launches managed RAG service with automatic "
        "chunking, embedding, and retrieval at $0.01 per query.",
        metadata={"days_ago": 2, "source": "aws_blog"}),
]

vs = Chroma.from_documents(articles, emb, collection_name="news_rag")
print(f"Indexed {len(articles)} articles")
for a in sorted(articles, key=lambda x: x.metadata["days_ago"]):
    print(f"  [{a.metadata['days_ago']:>3}d ago] {a.page_content[:55]}...")

## Step 3 — Standard Retrieval (No Freshness)

In [ ]:
query = "What's the latest news about AI startups and RAG?"
results = vs.similarity_search_with_score(query, k=3)
print(f"Query: {query}\n\nStandard retrieval (freshness-BLIND):")
for doc, score in results:
    print(f"  [{doc.metadata['days_ago']:>3}d ago] sim={score:.4f} — {doc.page_content[:55]}...")

## Step 4 — Freshness-Weighted Retrieval

Combine semantic similarity with exponential time-decay:
`score = (1-w) * similarity + w * exp(-days/τ)`

In [ ]:
def freshness_search(query, k=3, w=0.3, tau=30.0):
    """Blend semantic similarity with time recency."""
    all_results = vs.similarity_search_with_score(query, k=len(articles))
    scored = []
    for doc, sim_score in all_results:
        days = doc.metadata.get("days_ago", 180)
        freshness = math.exp(-days / tau)
        sim_norm = 1.0 / (1.0 + sim_score)  # lower distance = better
        combined = (1 - w) * sim_norm + w * freshness
        scored.append((doc, combined, sim_score, freshness))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]

print("Freshness-weighted retrieval:")
for doc, combined, sim, fresh in freshness_search(query):
    print(f"  [{doc.metadata['days_ago']:>3}d ago] combined={combined:.4f} "
          f"sim={sim:.4f} fresh={fresh:.3f}")

## Step 5 — Weight Sensitivity Analysis

In [ ]:
print(f"{'Weight':>8} {'Top-1 age':>10} {'Top-2 age':>10} {'Top-3 age':>10}")
print("-" * 42)
for w in [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]:
    results = freshness_search(query, k=3, w=w)
    ages = [f"{r[0].metadata['days_ago']}d" for r in results]
    print(f"{w:>8.1f} {ages[0]:>10} {ages[1]:>10} {ages[2]:>10}")

## Step 6 — Freshness-Aware QA Pipeline

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using the context. Mention when the information is from."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])
qa_chain = qa_prompt | llm | StrOutputParser()

fresh_results = freshness_search("Latest RAG news?", k=3, w=0.4)
ctx = "\n".join([f"[{r[0].metadata['days_ago']}d ago] {r[0].page_content}"
                 for r in fresh_results])
answer = qa_chain.invoke({"context": ctx, "question": "Latest RAG news?"})
print(f"Q: Latest RAG news?\nA: {answer}")

## Limitations

1. **Freshness bias:** Over-weighting may surface irrelevant recent docs.
2. **Decay tuning:** Optimal τ depends on domain (news: hours, research: months).
3. **No query classification:** Same weight applied to all queries.

## What You Learned

| Concept | What We Did |
|---|---|
| **Timestamped indexing** | Stored date metadata |
| **Freshness decay** | Exponential time-decay scoring |
| **Score blending** | Weighted similarity + freshness |
| **Weight sensitivity** | Effect on ranking |